In [83]:
import pandas as pd
pd.set_option("display.width", 200)

In [84]:
##############
# Import CSV
##############
df = pd.read_csv("../data/netflix_titles.csv")
# print(df.info())

In [85]:
############
# Extracting important data
############
netflix_df = df[["show_id", "type", "title" ,"country", "date_added", "release_year", "rating", "duration", "listed_in"]]
# netflix_df.head(5)

In [86]:
###########
# Amount of nulls in each column
###########
netflix_df.isna().sum().sort_values(ascending=False)

country         831
date_added       10
rating            4
duration          3
show_id           0
type              0
title             0
release_year      0
listed_in         0
dtype: int64

In [87]:
###########
# Cleaning country column
###########
netflix_df["country"] = netflix_df["country"].fillna("Unknown")

In [88]:
###########
# Splitting the country column into a new table. Better for data modeling and to avoid duplicated rows 
###########
# 194 show_id and 366 ili 193 365
country_df = netflix_df[["show_id", "country"]].copy()

country_df['country'] = country_df['country'].str.split(',').apply(
    lambda x: [i.strip() for i in x if i.strip()]
)

country_df = country_df.explode("country")

# print(country_df.groupby("country").size().to_string())

# Deleted country column netflix_df
netflix_df = netflix_df.drop(columns=["country"])

In [89]:
###########
# Moving values from the rating to the duration columns
###########
dirty_rating = netflix_df["rating"].str.contains("min", na=False)

netflix_df.loc[dirty_rating, "duration"] = netflix_df.loc[dirty_rating, "rating"]

netflix_df.loc[dirty_rating, "rating"] = pd.NA

###########
# Extract numeric values from the duration column and changing type to int
###########
netflix_df["duration"] = netflix_df["duration"].str.extract(r"(\d+)")

netflix_df["duration"] = netflix_df["duration"].astype("Int64")

In [90]:
###########
# Cleaning rating column
###########
netflix_df["rating"] = netflix_df["rating"].fillna("Unrated")


In [91]:
###########
# Trimming column date_added and changing data type to datetime
###########
netflix_df["date_added"] = netflix_df["date_added"].str.strip()

netflix_df["date_added"] = pd.to_datetime(netflix_df["date_added"], format="%B %d, %Y", errors="coerce")


In [94]:
###########
# Splitting the listed_in column into a new table
###########
genres_df = netflix_df[["show_id", "listed_in"]].copy()

genres_df["listed_in"] = genres_df["listed_in"].str.split(",")

genres_df = genres_df.explode("listed_in")

print(genres_df.to_string())



     show_id                      listed_in
0         s1                  Documentaries
1         s2         International TV Shows
1         s2                      TV Dramas
1         s2                   TV Mysteries
2         s3                 Crime TV Shows
2         s3         International TV Shows
2         s3          TV Action & Adventure
3         s4                     Docuseries
3         s4                     Reality TV
4         s5         International TV Shows
4         s5              Romantic TV Shows
4         s5                    TV Comedies
5         s6                      TV Dramas
5         s6                      TV Horror
5         s6                   TV Mysteries
6         s7       Children & Family Movies
7         s8                         Dramas
7         s8             Independent Movies
7         s8           International Movies
8         s9               British TV Shows
8         s9                     Reality TV
9        s10                    

In [93]:
# merged_df = country_df.merge(netflix_df, how="inner", on="show_id")
# merged_df = merged_df.set_index("show_id")

# merged_df = merged_df[merged_df["country"] == 'United Kingdom'][["title", "country", "type"]]

# merged_df.head(15)
